In [1]:
# ------------------------------------------------------------------------------
# 🛡️ AUTOMATED BENCHMARK PIPELINE: ON-OFF ATTACK (SLEEPING CELL) - FIXED ASR
# ------------------------------------------------------------------------------
# Cenário: Atacante alterna entre honesto e malicioso (S5-S9 e S15+).
# Correção ASR: Gatilho sutil (5.0) e avaliação apenas em amostras benignas.
# ------------------------------------------------------------------------------

import subprocess
import sys
import time
import os
import glob
import traceback
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist
from google.colab import drive

# Instalação Silenciosa
try:
    import hdbscan
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan"])
    import hdbscan

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
INPUT_PATTERN = "/content/drive/MyDrive/00_Tese_Benchmark/"
OUTPUT_DIR = "/content/drive/MyDrive/00_Tese_Benchmark/Benchmark_OnOff_Results_Fixed"

# Parâmetros Estatísticos
N_ROUNDS_STATISTICAL = 5
INITIAL_SEED = 42
NODES = ["Node1", "Node2", "Node3", "Node4"]
STAGES = [f"S{i}" for i in range(17)]

# Ataque On-Off (Sleeping Cell)
TRIGGER_VALUE = 5.0   # Trigger Sutil
POISON_RATE = 0.5     # 50% dos dados locais envenenados durante ataque

# Defesa (Adaptive)
ALPHA_PENALTY = 0.85
BETA_RECOVERY = 0.10
TRUST_THRESHOLD = 0.2
ABSOLUTE_MAJORITY_RULE = True
FLAME_NOISE_SIGMA = 0.001

# Modelo
HIDDEN_LAYERS = (64,)
LOCAL_EPOCHS = 3

# ==============================================================================
# 2. FUNÇÕES CORE
# ==============================================================================
def create_model(seed):
    return MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu',
                         solver='adam', random_state=seed, max_iter=1)

def get_weights(model):
    return np.concatenate([w.flatten() for w in model.coefs_] + [b.flatten() for b in model.intercepts_])

def set_weights(model, flat_weights, ref_model):
    idx = 0
    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size
    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx+size].reshape(shape)
        idx += size

def local_training(global_weights, X, y, classes, ref_model, seed):
    model = create_model(seed)
    if len(X) >= 5: model.partial_fit(X[:5], y[:5], classes=classes)
    else: model.partial_fit(X, y, classes=classes)
    set_weights(model, global_weights.copy(), ref_model)
    rng = np.random.RandomState(seed)
    for _ in range(LOCAL_EPOCHS):
        p = rng.permutation(len(X))
        model.partial_fit(X[p], y[p], classes=classes)
    return get_weights(model) - global_weights

# ==============================================================================
# 3. LÓGICA DE EXPERIMENTO (ON-OFF ATTACK)
# ==============================================================================
def run_single_seed_on_off(df, target_col, seed, dataset_name):
    y_raw = df[target_col].values.astype(int)
    X_raw = df.drop(columns=[target_col]).values
    classes = np.unique(y_raw)

    benign_class = 0
    target_attack_class = 1

    X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=seed)
    X_root, X_eval, y_root, y_eval = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test, random_state=seed)

    scaler = StandardScaler().fit(X_train)
    X_root_sc = scaler.transform(X_root)
    X_eval_sc = scaler.transform(X_eval)

    # --- SETUP DO BACKDOOR DE TESTE (ASR FIX) ---
    mask_benign = (y_eval == benign_class)
    if np.sum(mask_benign) > 0:
        X_backdoor = X_eval_sc[mask_benign].copy()
        if X_backdoor.shape[1] > 0:
            X_backdoor[:, -1] = TRIGGER_VALUE
    else:
        X_backdoor = np.empty((0, X_eval_sc.shape[1]))

    strategies = ["FedAvg", "Krum", "FLAME", "FLTrust", "Adaptive_Ultimate"]
    results = []

    # Configuração do Atacante (Sleeping Cell)
    SLEEPING_CELL_INDEX = [0] # Node 1 é o traidor

    print(f"\n   🌱 [Seed {seed}] Strategies: {', '.join(strategies)}")

    for strategy in strategies:
        g_model = create_model(seed)
        g_model.partial_fit(X_eval_sc[:10], y_eval[:10], classes=classes)
        g_weights = get_weights(g_model)
        trust_memory = {n: 0.5 for n in NODES}

        for stage_idx, stage_name in enumerate(STAGES):
            start_t = time.time()

            # --- CICLO ON-OFF ---
            # S0-S4: Honesto | S5-S9: Ataque | S10-S14: Honesto | S15+: Ataque
            is_attack_active = False
            if 5 <= stage_idx <= 9: is_attack_active = True
            elif stage_idx >= 15: is_attack_active = True

            shards_X = np.array_split(X_train, len(NODES))
            shards_y = np.array_split(y_train, len(NODES))
            local_updates = []

            for i, node in enumerate(NODES):
                X_n = shards_X[i].copy(); y_n = shards_y[i].copy()

                if i in SLEEPING_CELL_INDEX and is_attack_active:
                    n_p = int(len(y_n) * POISON_RATE)
                    if n_p > 0:
                        rng = np.random.RandomState(seed + stage_idx + i)
                        p_idx = rng.choice(len(y_n), n_p, replace=False)
                        if X_n.shape[1] > 0: X_n[p_idx, -1] = TRIGGER_VALUE
                        y_n[p_idx] = target_attack_class

                delta = local_training(g_weights, scaler.transform(X_n), y_n, classes, g_model, seed)
                local_updates.append(delta)

            # Agregação
            root_delta = local_training(g_weights, X_root_sc, y_root, classes, g_model, seed)
            root_norm = np.linalg.norm(root_delta) + 1e-12
            agg_update = np.zeros_like(g_weights)

            if strategy == "FedAvg":
                agg_update = np.mean(local_updates, axis=0)
            elif strategy == "Krum":
                k = max(1, len(local_updates) - 3)
                dists = cdist(np.stack(local_updates), np.stack(local_updates))
                scores = [np.sum(np.sort(d)[1:k+1]) for d in dists]
                agg_update = local_updates[np.argmin(scores)]
            elif strategy == "FLAME":
                stk = np.stack(local_updates)
                dists = cosine_distances(stk)
                labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=2, allow_single_cluster=True).fit_predict(dists)
                u, c = np.unique(labels, return_counts=True)
                valid = u[u != -1]
                sel = stk[np.where(labels == valid[np.argmax(c[u != -1])])[0]] if len(valid) > 0 else stk
                med = np.median(np.linalg.norm(sel, axis=1))
                clip = [v * min(1, med/(np.linalg.norm(v)+1e-12)) for v in sel]
                agg_update = np.mean(clip, axis=0) + np.random.normal(0, FLAME_NOISE_SIGMA * med, g_weights.shape)
            elif strategy == "FLTrust":
                scores = []
                for d in local_updates:
                    sim = np.dot(root_delta, d) / (root_norm * (np.linalg.norm(d) + 1e-12))
                    scores.append(max(0, sim))
                    agg_update += max(0, sim) * (d * (root_norm / (np.linalg.norm(d) + 1e-12)))
                if sum(scores) > 0: agg_update /= sum(scores)
            elif strategy == "Adaptive_Ultimate":
                valid_cnt = 0
                maj_thr = (len(NODES) // 2) + 1
                trust_w = []
                for i, d in enumerate(local_updates):
                    sim = np.dot(root_delta, d) / (root_norm * (np.linalg.norm(d) + 1e-12))
                    old = trust_memory[NODES[i]]
                    if sim < 0: new = 0.0 # Punição Imediata
                    elif (sim - old) < 0: new = old + (ALPHA_PENALTY * (sim - old))
                    else: new = old + (BETA_RECOVERY * (sim - old)) # Recuperação Lenta
                    new = np.clip(new, 0.0, 1.0)
                    trust_memory[NODES[i]] = new
                    if new < TRUST_THRESHOLD: w = 0.0
                    else: w, valid_cnt = new, valid_cnt + 1
                    trust_w.append(w)
                if valid_cnt < maj_thr:
                    agg_update = np.zeros_like(g_weights)
                else:
                    tot = 0
                    for i, w in enumerate(trust_w):
                        if w > 0:
                            agg_update += w * (local_updates[i] * (root_norm / (np.linalg.norm(local_updates[i]) + 1e-12)))
                            tot += w
                    if tot > 0: agg_update /= tot

            g_weights += agg_update
            set_weights(g_model, g_weights, g_model)

            # --- AVALIAÇÃO CORRIGIDA ---
            f1 = f1_score(y_eval, g_model.predict(X_eval_sc), average='macro')

            # ASR apenas em Benignos
            if len(X_backdoor) > 0:
                asr = np.mean(g_model.predict(X_backdoor) == target_attack_class)
            else:
                asr = 0.0

            results.append({"Seed": seed, "Strategy": strategy, "Stage": stage_idx, "F1": f1, "ASR": asr, "Time": time.time() - start_t})

        final_res = results[-1]
        print(f"      ✅ {strategy}: F1={final_res['F1']:.4f} | ASR={final_res['ASR']:.4f}")

    return results

# ==============================================================================
# 4. ORQUESTRADOR
# ==============================================================================
def process_all_datasets_on_off():
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    all_files = glob.glob(INPUT_PATTERN + "*.csv")
    files = [f for f in all_files if "Benchmark" not in os.path.basename(f) and "RESULTADOS" not in os.path.basename(f)]

    if not files: print("❌ Nenhum dataset válido encontrado."); return

    print(f"📦 INICIANDO ON-OFF ATTACK (FIXED ASR): {len(files)} Datasets.\n")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    global_summary_list = []

    for idx, fpath in enumerate(files):
        fname = os.path.basename(fpath); dataset_name = fname.replace(".csv", "")
        print("="*80); print(f"▶️  [{idx+1}/{len(files)}] {dataset_name}"); print("="*80)

        try:
            df = pd.read_csv(fpath)
            cols_ignore = ['Dataset_Source', 'device_category', 'host', 'ts', 'date', 'time']
            df = df.drop(columns=[c for c in cols_ignore if c in df.columns], errors='ignore')

            target_col = None
            for c in df.columns:
                if c.lower() in ['class', 'target', 'label', 'attack', 'benign']: target_col = c; break
            if target_col is None: target_col = df.columns[-1]

            if df[target_col].dtype == object: df[target_col] = LabelEncoder().fit_transform(df[target_col].astype(str))

            dataset_results = []
            for i in range(N_ROUNDS_STATISTICAL):
                seed = INITIAL_SEED + i
                res = run_single_seed_on_off(df, target_col, seed, dataset_name)
                dataset_results.extend(res)

            # Salvamento
            dataset_dir = os.path.join(OUTPUT_DIR, dataset_name)
            os.makedirs(dataset_dir, exist_ok=True)

            full_df = pd.DataFrame(dataset_results)
            full_df.to_csv(os.path.join(dataset_dir, "Raw_Results_OnOff.csv"), index=False)

            summary = full_df.groupby(['Strategy', 'Stage']).agg({'F1':['mean','std'], 'ASR':['mean','std']}).reset_index()
            summary.columns = ['Strategy','Stage','F1_Mean','F1_Std','ASR_Mean','ASR_Std']
            summary.to_csv(os.path.join(dataset_dir, "Summary_Stats_OnOff.csv"), index=False)

            plt.figure(figsize=(10, 6))
            sns.lineplot(data=full_df, x='Stage', y='ASR', hue='Strategy', errorbar='sd')
            # Marcar fases de ataque
            plt.axvspan(5, 9, color='red', alpha=0.1, label='Attack')
            plt.axvspan(15, 16, color='red', alpha=0.1)
            plt.title(f'ASR On-Off Attack - {dataset_name}')
            plt.savefig(os.path.join(dataset_dir, "Plot_OnOff.png"))
            plt.close()

            last = summary[summary['Stage'] == 16].copy(); last['Dataset'] = dataset_name
            global_summary_list.append(last)
            print(f"   ✨ Sucesso: {dataset_name}\n")

        except Exception as e:
            print(f"   ❌ ERRO: {str(e)}\n"); continue

    if global_summary_list:
        pd.concat(global_summary_list).to_csv(os.path.join(OUTPUT_DIR, "RELATORIO_ONOFF_FINAL.csv"), index=False)
        print("\n🏆 Benchmark On-Off Finalizado!")

if __name__ == "__main__":
    process_all_datasets_on_off()

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


Mounted at /content/drive
📦 INICIANDO ON-OFF ATTACK (FIXED ASR): 1 Datasets.

▶️  [1/1] Ecobee_Thermostat_mirai_attacks_consolidado

   🌱 [Seed 42] Strategies: FedAvg, Krum, FLAME, FLTrust, Adaptive_Ultimate
      ✅ FedAvg: F1=0.9994 | ASR=0.0015
      ✅ Krum: F1=0.9996 | ASR=0.0008
      ✅ FLAME: F1=0.9992 | ASR=0.0008
      ✅ FLTrust: F1=0.9992 | ASR=0.0015
      ✅ Adaptive_Ultimate: F1=0.9996 | ASR=0.0023

   🌱 [Seed 43] Strategies: FedAvg, Krum, FLAME, FLTrust, Adaptive_Ultimate
      ✅ FedAvg: F1=0.9994 | ASR=0.0000
      ✅ Krum: F1=0.9975 | ASR=0.0015
      ✅ FLAME: F1=0.9994 | ASR=0.0008
      ✅ FLTrust: F1=0.9996 | ASR=0.0008
      ✅ Adaptive_Ultimate: F1=0.9994 | ASR=0.0008

   🌱 [Seed 44] Strategies: FedAvg, Krum, FLAME, FLTrust, Adaptive_Ultimate
      ✅ FedAvg: F1=0.9998 | ASR=0.0000
      ✅ Krum: F1=0.9998 | ASR=0.0008
      ✅ FLAME: F1=0.9998 | ASR=0.0008
      ✅ FLTrust: F1=0.9986 | ASR=0.0000
      ✅ Adaptive_Ultimate: F1=1.0000 | ASR=0.0000

   🌱 [Seed 45] Strategies: 